# 99. Finding example sites -- PV/Load phase-count x conformance

A pure site-finding tool for Section 5-style plotting, not part of the blind
pipeline itself: which sites are

* 3-phase PV + 3-phase Load
* 1-phase PV + 3-phase Load
* 3-phase PV + 1-phase Load
* 1-phase PV + 1-phase Load

each split into **conformant**/**non-conformant**, so you have concrete
`site_id`s to hand `ConformancePlot.plot_site_day_conformance`/
`plot_pv_circuit_vvar_scatter` in notebook 05.

Two things are deliberately NOT the blind pipeline's own numbers:

1. **Phase configuration** comes from `ami_circuit_metadata`'s real
   `circuit_type` (`pv_site_net`/`ac_load_net`) -- the true, ground-truth
   circuit labels, not
   `pv_mapping`'s blind inference. This notebook is a grading/site-finding
   tool (like notebook 05's own Section 7), so using the true labels here
   doesn't compromise anything -- nothing here feeds back into the blind
   pipeline.
2. **Conformant/non-conformant** comes straight from the ORIGINAL Solar
   Analytics / Athena conformance tables (`conformance_voltvar`/
   `conformance_voltwatt` from `TABLES`), reusing the exact methodology
   notebook 05 Section 8 already validated -- NOT this project's own
   from-`ami_meter_residential` recalculation (Sections 6/7 there). See the
   `FLEX_SELECTION` note in Section 2 below for the one non-obvious setting
   this depends on.

**Requires AWS SSO login in your own terminal first**
(`aws sso login --profile ciccada`) -- this notebook cannot do that for you.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import pandas as pd

from bms_sa_review.data_query.lib.conformance_metrics import aggregate_sites
from bms_sa_review.shared.ciccada_config import AS4777, SAI, TABLES
from bms_sa_review.synthetic_ami_creation.config import ami_config as Config
from bms_sa_review.synthetic_ami_creation.lib import ami_athena as Athena

pd.set_option("display.max_columns", None)


## 0. Local store -- residential site/circuit metadata + phase-count source

No polarity correction here (unlike notebook 05's bootstrap) -- this
notebook never computes conformance from `ami_meter_residential` itself.
That table is loaded only to confirm which circuits actually have landed
LOAD data (so every `site_id` this notebook hands back can actually be
plotted with `plot_site_day_conformance`). It is NOT used for phase
counting -- `ami_meter`/`ami_meter_clean`/`ami_meter_residential` are
built by `ami_build.build_ami_meter`, which keeps ONLY `circuit_type ==
"ac_load_net"` rows; no PV circuit ever lands there. Phase counts (both PV
and load) instead come from `ami_raw_phaseseparate` -- the one Phase-5
deliverable that is circuit-preserving AND keeps both `ac_load_net` and
`pv_site_net` rows, tagged by `circuit_type` (see Section 1 below). It is
NOT itself scoped to residential/small-PV sites, so it is restricted to
`site_meta`'s site_id set here.


In [ ]:
con = duckdb.connect()

con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_residential AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_meter_residential").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")
con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata_residential AS
    SELECT * FROM read_parquet('{Config.store_path("ami_site_metadata_residential").as_posix()}')
""")
con.sql(f"""
    CREATE OR REPLACE VIEW ami_circuit_metadata AS
    SELECT * FROM read_parquet('{Config.store_path("ami_circuit_metadata").as_posix()}')
""")
# Circuit-preserving, BOTH load ("ac_load_net") AND PV ("pv_site_net") rows,
# tagged by circuit_type -- unlike ami_meter*, which is load-only. Not
# residential-scoped on disk, so filtered to site_meta's site_id below.
con.sql(f"""
    CREATE OR REPLACE VIEW ami_raw_phaseseparate AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_raw_phaseseparate").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")

site_meta = con.sql("SELECT * FROM ami_site_metadata_residential").df()
print(f"{len(site_meta):,} residential sites loaded.")


## 1. Ground-truth phase configuration per site

Per site: how many DISTINCT circuits are truly PV (`circuit_type ==
'pv_site_net'`) and how many are truly load (`circuit_type ==
'ac_load_net'`), from `ami_raw_phaseseparate` -- restricted to
`site_meta`'s residential site_ids (this table is not itself
residential-scoped on disk).

**Why not `ami_meter_residential` (as an earlier version of this notebook
did)?** `ami_meter`/`ami_meter_clean`/`ami_meter_residential` are built by
`ami_build.build_ami_meter`, which keeps ONLY `circuit_type ==
"ac_load_net"` rows -- it is the "net meter" table by design (that's the
whole premise of notebook 05's blind pipeline: PV isn't directly present in
the meter data, it has to be inferred). Restricting to circuits landed in
`ami_meter_residential` therefore silently excluded every PV circuit,
regardless of how `is_pv_circuit`/PV `circuit_type` was matched -- `n_pv_circuits`
came back 0 for every site no matter what. `ami_raw_phaseseparate` is the
Phase-5 deliverable built specifically to keep BOTH sides
(`build_ami_raw_phaseseparate`), so it's the correct ground-truth source
for phase counts.


In [ ]:
circuit_meta_residential = con.sql("""
    SELECT DISTINCT r.site_id, r.circuit_id, r.circuit_type
    FROM ami_raw_phaseseparate r
    INNER JOIN (SELECT DISTINCT site_id FROM ami_site_metadata_residential) residential
        USING (site_id)
""").df()

phase_counts = (
    circuit_meta_residential
    .assign(
        is_load_circuit=lambda d: d.circuit_type == "ac_load_net",
        is_pv_circuit=lambda d: d.circuit_type == "pv_site_net",
    )
    .groupby("site_id")
    .agg(n_pv_circuits=("is_pv_circuit", "sum"), n_load_circuits=("is_load_circuit", "sum"))
    .reset_index()
)
# Sites in site_meta with NO row at all in ami_raw_phaseseparate (e.g. never
# landed there) still get a row, both counts 0 -- kept, not dropped, so the
# denominator below (all residential sites) stays meaningful.
phase_counts = site_meta[["site_id", "n_pv_phases", "n_load_phases"]].merge(
    phase_counts, on="site_id", how="left",
)
phase_counts[["n_pv_circuits", "n_load_circuits"]] = (
    phase_counts[["n_pv_circuits", "n_load_circuits"]].fillna(0).astype(int)
)

print(f"{circuit_meta_residential.site_id.nunique():,} of {len(site_meta):,} residential sites "
      "have at least one row in ami_raw_phaseseparate.")
print("Observed PV-circuit count per site (ground truth, from ami_raw_phaseseparate):")
display(phase_counts.n_pv_circuits.value_counts().sort_index())
print("Observed load-circuit count per site (ground truth, from ami_raw_phaseseparate):")
display(phase_counts.n_load_circuits.value_counts().sort_index())

# ami_metadata.py's own caveat: n_pv_phases/n_load_phases (site-level
# metadata) are computed against a possibly-different, earlier circuit_meta
# scope than what actually landed here -- a mismatch isn't necessarily a
# bug, just a reason to trust the OBSERVED count (above) over the metadata
# one when the goal is "can I actually plot this site this way".
mismatch = phase_counts.loc[
    (phase_counts.n_pv_circuits != phase_counts.n_pv_phases.fillna(0))
    | (phase_counts.n_load_circuits != phase_counts.n_load_phases.fillna(0))
]
print(f"{len(mismatch):,} of {len(phase_counts):,} sites have an observed-vs-metadata "
      "phase-count mismatch (informational only -- observed count is used below).")


In [ ]:
PHASE_COMBOS = {
    "3-phase PV + 3-phase Load": (3, 3),
    "1-phase PV + 3-phase Load": (1, 3),
    "3-phase PV + 1-phase Load": (3, 1),
    "1-phase PV + 1-phase Load": (1, 1),
}


def phase_combo_label(n_pv, n_load):
    for label, (pv, load) in PHASE_COMBOS.items():
        if n_pv == pv and n_load == load:
            return label
    return "other"


phase_counts["phase_combo"] = phase_counts.apply(
    lambda r: phase_combo_label(r.n_pv_circuits, r.n_load_circuits), axis=1,
)
print("Residential sites per phase combo (observed, ground truth):")
display(phase_counts.phase_combo.value_counts().reindex(list(PHASE_COMBOS) + ["other"], fill_value=0))


## 2. Historical (original Solar Analytics / Athena) conformance per site

Reuses notebook 05 Section 8's already-confirmed methodology and settings
-- not a new calculation. One setting is NOT this project's usual default
and matters a lot here: **`FLEX_SELECTION = "include"`**. `AnalysisConfig`'s
project-wide default is `"exclude"`, but that returns ZERO rows against
these two tables -- the single methodology combination actually stored in
them (confirmed in notebook 05 Section 8.2/8.3) has `flex_selection =
"include"`. Pull the FULL `HISTORICAL_YEARS` here (not scoped to one
month), since the point of this notebook is to find candidates across as
much data as exists, not to match a narrower scope elsewhere.


In [ ]:
Athena.require_credentials()
print(Athena.credential_status())

HISTORICAL_YEARS = (2024, 2025)
RATING_BASIS = "ac_capacity_kw"
CAPABILITY_PROFILE = "review_corrected"       # Volt-VAr only
VOLTAGE_AGGREGATION = "avg"
FLEX_SELECTION = "include"                    # see markdown above -- NOT the
                                               # project's usual "exclude" default
SITE_NONCONF_THRESHOLD = AS4777["SITE_CONF_THRESH"]

all_site_ids = ",".join(str(int(s)) for s in site_meta.site_id.tolist())
year_list = ", ".join(str(y) for y in HISTORICAL_YEARS)

print("Historical pull -- methodology used:")
display(pd.DataFrame([
    {"setting": "years", "value": ", ".join(map(str, HISTORICAL_YEARS))},
    {"setting": "rating_basis", "value": RATING_BASIS},
    {"setting": "capability_profile (Volt-VAr only)", "value": CAPABILITY_PROFILE},
    {"setting": "voltage_aggregation", "value": VOLTAGE_AGGREGATION},
    {"setting": "flex_selection", "value": FLEX_SELECTION},
]))


In [ ]:
historical_vvar_raw = Athena.aq(f"""
    SELECT site_id,
           sum(Q_adverse_count) AS adverse,
           sum(Q_inactive_count) AS inactive,
           sum(Q_significant_shortfall_count) AS shortfall,
           sum(total_count) AS capability_assessable_count
    FROM {TABLES["conformance_voltvar"]}
    WHERE year IN ({year_list})
      AND rating_basis = '{RATING_BASIS}'
      AND capability_profile = '{CAPABILITY_PROFILE}'
      AND voltage_aggregation = '{VOLTAGE_AGGREGATION}'
      AND flex_selection = '{FLEX_SELECTION}'
      AND site_id IN ({all_site_ids})
    GROUP BY site_id
""", database=SAI, label="historical_voltvar_site_all")
historical_vvar_raw["reduced_nonconf_count"] = (
    historical_vvar_raw[["adverse", "inactive", "shortfall"]].sum(axis=1)
)
historical_vvar_site = aggregate_sites(
    historical_vvar_raw, "reduced_nonconf_count", "capability_assessable_count",
    threshold=SITE_NONCONF_THRESHOLD,
)[["site_id", "conformant", "nonconf_frac"]].rename(
    columns={"conformant": "voltvar_conformant", "nonconf_frac": "voltvar_nonconf_frac"}
)

historical_vw_raw = Athena.aq(f"""
    SELECT site_id,
           sum(nonconformance_voltwatt_count) AS nonconforming_count,
           sum(total_count) AS exposed_count
    FROM {TABLES["conformance_voltwatt"]}
    WHERE year IN ({year_list})
      AND rating_basis = '{RATING_BASIS}'
      AND voltage_aggregation = '{VOLTAGE_AGGREGATION}'
      AND flex_selection = '{FLEX_SELECTION}'
      AND site_id IN ({all_site_ids})
    GROUP BY site_id
""", database=SAI, label="historical_voltwatt_site_all")
historical_vw_site = aggregate_sites(
    historical_vw_raw, "nonconforming_count", "exposed_count", threshold=SITE_NONCONF_THRESHOLD,
)[["site_id", "conformant", "nonconf_frac"]].rename(
    columns={"conformant": "voltwatt_conformant", "nonconf_frac": "voltwatt_nonconf_frac"}
)

print(f"{len(historical_vvar_site):,} sites with a historical Volt-VAr verdict, "
      f"{len(historical_vw_site):,} with a historical Volt-Watt verdict.")
Athena.scan_report()


## 3. Combined catalog -- phase combo x conformance

One row per residential site: observed phase combo (Section 1) plus the
historical Volt-VAr/Volt-Watt conformant flag (Section 2), wherever both
are available.


In [ ]:
site_catalog = (
    phase_counts[["site_id", "n_pv_circuits", "n_load_circuits", "phase_combo"]]
    .merge(historical_vvar_site, on="site_id", how="left")
    .merge(historical_vw_site, on="site_id", how="left")
)

print(
    f"{len(site_catalog):,} residential sites in the combined catalog -- "
    f"{site_catalog.voltvar_conformant.notna().sum():,} have a historical Volt-VAr verdict, "
    f"{site_catalog.voltwatt_conformant.notna().sum():,} have a historical Volt-Watt verdict."
)


def summarize(mechanism_column, label):
    summary = (
        site_catalog.dropna(subset=[mechanism_column])
        .groupby(["phase_combo", mechanism_column])
        .size()
        .unstack(fill_value=0)
        .rename(columns={True: "conformant", False: "non_conformant"})
        .reindex(index=list(PHASE_COMBOS) + ["other"], fill_value=0)
    )
    print(f"{label} -- sites available per phase combo x conformance:")
    display(summary)
    return summary


voltvar_summary = summarize("voltvar_conformant", "Volt-VAr")
voltwatt_summary = summarize("voltwatt_conformant", "Volt-Watt")


## 4. Example `site_id`s for each combination

Prints up to 5 example `site_id`s for every (phase combo x mechanism x
conformant/non-conformant) cell -- a "0 found" is informative on its own
(e.g. "3-phase PV + 1-phase Load" is physically unusual and may genuinely
be empty or "other"-bucketed if the numbers don't line up as expected).


In [ ]:
def example_sites(mechanism_column, phase_combo, conformant, n=10):
    matches = site_catalog.loc[
        (site_catalog.phase_combo == phase_combo) & (site_catalog[mechanism_column] == conformant)
    ]
    return matches.sort_values("site_id").site_id.astype(int).tolist()[:n]


for mechanism_column, mech_label in [("voltvar_conformant", "Volt-VAr"), ("voltwatt_conformant", "Volt-Watt")]:
    print(f"=== {mech_label} ===")
    for combo in PHASE_COMBOS:
        for conformant, tag in [(True, "conformant"), (False, "non-conformant")]:
            ids = example_sites(mechanism_column, combo, conformant, n=50)
            preview = ids[:5]
            print(f"  {combo:<28} {tag:<15} {len(ids):>4} found  -- e.g. {preview}")
    print()


## 5. Pick one -- candidate list for a single combination

Set the three values below and re-run this cell to get a fuller candidate
list for one specific combination, ready to copy into notebook 05's
`SITE_ID = ...` cells.


In [ ]:
MECHANISM = "voltvar_conformant"              # or "voltwatt_conformant"
PHASE_COMBO = "3-phase PV + 3-phase Load"      # one of PHASE_COMBOS' keys
WANT_CONFORMANT = True                        # True = conformant, False = non-conformant

candidates = example_sites(MECHANISM, PHASE_COMBO, WANT_CONFORMANT, n=50)
print(
    f"{len(candidates)} candidate site_ids for {PHASE_COMBO!r} / {MECHANISM} / "
    f"{'conformant' if WANT_CONFORMANT else 'non-conformant'}:"
)
candidates
